In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import RandomForestRegressor

In [2]:
# Read the dataset.
df = pd.read_csv("winequality-white.csv", sep=";")

# Separate features and target.
X = df.drop(columns="quality")
y = df["quality"]

# Reserve 20% for testing.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Reserve 10% of the full dataset for validation.
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.125,
    random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

Training samples: 3428
Validation samples: 490
Test samples: 980


In [3]:
scaler = StandardScaler()

# Learn the scaling parameters from training data only.
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaling to validation data.
X_val_scaled = scaler.transform(X_val)

print("Training shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)

Training shape: (3428, 11)
Validation shape: (490, 11)


In [4]:
knn_grid = {
    "n_neighbors": list(range(1, 101)),
    "weights": ["uniform", "distance"],
    "p": [1, 2, 3]
}

knn_configs = list(ParameterGrid(knn_grid))

print("Number of configurations:", len(knn_configs))
print("Number of validation samples:", len(X_val))

assert len(knn_configs) > len(X_val)

Number of configurations: 600
Number of validation samples: 490


In [5]:
best_knn = None
best_knn_params = None
best_knn_val_mse = float("inf")

knn_records = []

for i, params in enumerate(knn_configs, start=1):
    # Create and fit the candidate model.
    model = KNeighborsRegressor(**params)
    model.fit(X_train_scaled, y_train)

    # Evaluate on the validation set.
    val_pred = model.predict(X_val_scaled)
    val_mse = mean_squared_error(y_val, val_pred)

    # Record this configuration's result.
    knn_records.append({
        **params,
        "validation_mse": val_mse
    })

    # Keep the model with the lowest validation MSE.
    if val_mse < best_knn_val_mse:
        best_knn = model
        best_knn_params = params.copy()
        best_knn_val_mse = val_mse

    if i % 50 == 0:
        print(f"Completed {i}/{len(knn_configs)}", flush=True)

knn_results = pd.DataFrame(knn_records)

print("\nBest parameters:", best_knn_params)
print(f"Best validation MSE: {best_knn_val_mse:.6f}")

knn_results.sort_values("validation_mse").head()

Completed 50/600
Completed 100/600
Completed 150/600
Completed 200/600
Completed 250/600
Completed 300/600
Completed 350/600
Completed 400/600
Completed 450/600
Completed 500/600
Completed 550/600
Completed 600/600

Best parameters: {'n_neighbors': 25, 'p': 1, 'weights': 'distance'}
Best validation MSE: 0.298739


,n_neighbors,p,weights,validation_mse
145,25,1,distance,0.298739
109,19,1,distance,0.299662
103,18,1,distance,0.299994
163,28,1,distance,0.300058
97,17,1,distance,0.300187


In [6]:
rf_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_leaf": [1, 2, 3, 4, 5, 6, 8, 10, 12, 16],
    "max_features": [0.5, 0.75, 1.0]
}

rf_configs = list(ParameterGrid(rf_grid))

print("Random Forest configurations:", len(rf_configs))
print("Validation samples:", len(X_val))

assert len(rf_configs) > len(X_val)

Random Forest configurations: 540
Validation samples: 490


In [7]:
best_rf = None
best_rf_params = None
best_rf_val_mse = float("inf")

rf_records = []

for i, params in enumerate(rf_configs, start=1):
    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    # Train using the training set only.
    model.fit(X_train, y_train)

    # Select hyperparameters using validation MSE.
    val_pred = model.predict(X_val)
    val_mse = mean_squared_error(y_val, val_pred)

    rf_records.append({
        **params,
        "validation_mse": val_mse
    })

    if val_mse < best_rf_val_mse:
        best_rf = model
        best_rf_params = params.copy()
        best_rf_val_mse = val_mse

    if i % 20 == 0:
        print(
            f"Completed {i}/{len(rf_configs)}",
            flush=True
        )

rf_results = pd.DataFrame(rf_records)

print("\nBest Random Forest parameters:", best_rf_params)
print(f"Best validation MSE: {best_rf_val_mse:.6f}")

rf_results.sort_values("validation_mse").head()

Completed 20/540
Completed 40/540
Completed 60/540
Completed 80/540
Completed 100/540
Completed 120/540
Completed 140/540
Completed 160/540
Completed 180/540
Completed 200/540
Completed 220/540
Completed 240/540
Completed 260/540
Completed 280/540
Completed 300/540
Completed 320/540
Completed 340/540
Completed 360/540
Completed 380/540
Completed 400/540
Completed 420/540
Completed 440/540
Completed 460/540
Completed 480/540
Completed 500/540
Completed 520/540
Completed 540/540

Best Random Forest parameters: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 100}
Best validation MSE: 0.308704


,max_depth,max_features,min_samples_leaf,n_estimators,validation_mse
1,NaN,0.50,1,100,0.308704
451,30.0,0.50,1,100,0.309296
2,NaN,0.50,1,200,0.310285
452,30.0,0.50,1,200,0.310865
480,30.0,0.75,1,50,0.310978


In [8]:
# -------------------------
# Final evaluation on the test set
# -------------------------

# Use the scaler previously fitted on training data.
X_test_scaled = scaler.transform(X_test)

# Evaluate the selected KNN.
knn_test_pred = best_knn.predict(X_test_scaled)
knn_test_mse = mean_squared_error(y_test, knn_test_pred)

# Evaluate the selected Random Forest.
rf_test_pred = best_rf.predict(X_test)
rf_test_mse = mean_squared_error(y_test, rf_test_pred)

# Summarize the results.
summary = pd.DataFrame({
    "Model": ["KNN", "Random Forest"],
    "Configurations": [len(knn_configs), len(rf_configs)],
    "Validation MSE": [best_knn_val_mse, best_rf_val_mse],
    "Test MSE": [knn_test_mse, rf_test_mse]
})

summary["Test - Validation"] = (
    summary["Test MSE"] - summary["Validation MSE"]
)

summary.round(6)

,Model,Configurations,Validation MSE,Test MSE,Test - Validation
0,KNN,600,0.298739,0.368550,0.069811
1,Random Forest,540,0.308704,0.348184,0.039480
